# TRADES episodes on a GPU (10 diffusion sampling steps)

Needs two datasets: `trades-bundle` (order-book data, from the training run) and `trades-sim-bundle` (checkpoint, patch, scripts, job list). Writes `trades_runs.zip` to the output folder; episodes finished before the time limit are kept.


In [ ]:
import glob, os, shutil, subprocess, sys
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found: attach the dataset"
    return hits[0]
SIM = os.path.dirname(find("run_sim_jobs.py"))
DATA = [d for d in glob.glob("/kaggle/input/**/INTC_*_*", recursive=True) if os.path.isdir(d)][0]
OUT, WORK = "/kaggle/working", "/tmp/trades_sim"
os.makedirs(WORK, exist_ok=True)
import torch
print(SIM, DATA, "GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
os.chdir(WORK)
if not os.path.isdir("DeepMarket"):
    subprocess.run("git clone -q https://github.com/LeonardoBerti00/DeepMarket.git", shell=True, check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
os.chdir(f"{WORK}/DeepMarket")
subprocess.run("git checkout -q 8f1f89b7285c79a73f528b88b60f74ce58faadc4 && git checkout -q -- .", shell=True, check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
subprocess.run(f"git apply --whitespace=nowarn {SIM}/deepmarket_simbench.patch", shell=True, check=True)
shutil.copy(f"{SIM}/constants.py", "constants.py")  # the training run's constants (dates and split)
subprocess.run(f"{sys.executable} -m pip install -q lightning einops torch_ema lion_pytorch wandb", shell=True, check=True)
dst = f"{WORK}/DeepMarket/data/INTC/{os.path.basename(DATA)}"
shutil.rmtree(f"{WORK}/DeepMarket/data/INTC", ignore_errors=True)
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copytree(DATA, dst)
for f in ("run_sim_jobs.py", "run_trades_episode.py"):
    shutil.copy(f"{SIM}/{f}", f"{WORK}/{f}")
print("ready")


In [ ]:
# all episodes; finished ones are written to OUT/runs as they complete
ckpt = glob.glob(f"{SIM}/*.ckpt")[0]
p = subprocess.Popen(f"{sys.executable} -u {WORK}/run_sim_jobs.py {WORK}/DeepMarket {ckpt} {SIM}/jobs.txt {OUT}/runs --procs 4 --hours 10.3 --device cuda,cpu",
                     shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
print("exit code", p.wait())


In [ ]:
shutil.make_archive(f"{OUT}/trades_runs", "zip", f"{OUT}/runs")
shutil.rmtree(f"{OUT}/runs")  # keep the saved output to one file
print(os.listdir(OUT))
